 # PolicyProof AI — Stage 3: AlignTune QLoRA T4

Research-only integration test. Run in a **new Google Colab T4 notebook**. Use your downloaded CuratorKIT ZIP, which contains only fictional training rows. The previous 48-case benchmark must never enter training. **Installation and GPU training default to OFF.** This is not a verified model improvement. Tiny Aya's CC-BY-NC license applies.

## 1. Check T4 and pull versioned project code.

In [ ]:
import torch, subprocess, pathlib, os, sys, json, time
assert torch.cuda.is_available(), "Select a T4 GPU under Runtime settings and connect."
print("GPU:", torch.cuda.get_device_name(0))
ROOT=pathlib.Path("/content/PolicyProof-AI")
if not ROOT.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/betterlifestyle04-commits/PolicyProof-AI.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"pull","--ff-only"],check=True)
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
REV=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
print("Code revision:",REV)


## 2. Upload the CuratorKIT training ZIP and verify its integrity before extraction. Only allowlisted files are extracted.

In [ ]:
from google.colab import files
from zipfile import ZipFile
from scripts.verify_curator_archive import check_zip
uploaded=files.upload()
assert len(uploaded)==1, "Upload only the policyproof_curatorkit_train_v1.zip"
zip_name=next(iter(uploaded))
assert zip_name.endswith(".zip")
report=check_zip(zip_name)
assert report["status"]=="pass" and report["row_count"]==78
print(json.dumps(report,indent=2,ensure_ascii=False))
CURATED=ROOT/"artifacts"/"verified_curator_train"
CURATED.mkdir(parents=True,exist_ok=True)
allowed={"sft_alpaca.jsonl","sft_sharegpt.jsonl","manifest.json",
         "lexsi_provenance.json","checksums.txt","README.md","dataset_card.md","rejected.jsonl"}
with ZipFile(zip_name) as z:
    for name in sorted(allowed.intersection(z.namelist())):
        (CURATED/name).write_bytes(z.read(name))


## 3. Authenticate Hugging Face and record the exact model revision

Use your approved Hugging Face account and a read-only token via the login widget if necessary. Do not paste credentials into a notebook cell. This step does not download model weights.

In [ ]:
from huggingface_hub import whoami, notebook_login, model_info
try:
    hf_account = whoami()
except Exception:
    notebook_login()
    hf_account = whoami()
MODEL_ID = "CohereLabs/tiny-aya-global"
MODEL_SHA = model_info(MODEL_ID).sha
assert isinstance(MODEL_SHA, str) and len(MODEL_SHA) >= 12
print("HF account:", hf_account["name"])
print("Approved model:", MODEL_ID)
print("Model commit SHA:", MODEL_SHA)


## 4. Verify AlignTune and the actual training dependencies

The `pip check` output can include unrelated preinstalled Colab packages (Google ADK, RAPIDS, IPython). Do not downgrade those blindly. This cell prints all `pip check` details, then runs a **separate-process smoke test** of AlignTune's trainer factory, Transformers, PEFT, TRL, BitsAndBytes and CUDA.

A successful import test is only a prerequisite; it is **not** proof Tiny Aya's trainer configuration will work or fit in VRAM. Keep `RUN_TRAINING=False` until the test is reviewed.

In [ ]:
import subprocess, sys, json
from importlib.metadata import version, PackageNotFoundError

INSTALL_NOW = False
if INSTALL_NOW:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "aligntune>=0.1.10,<1", "huggingface-hub>=1.31,<2"],
        check=True
    )

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True, text=True
)
print("GLOBAL PIP CHECK STATUS:", check.returncode)
print(check.stdout or check.stderr or "(no output)")

for package in ["aligntune", "transformers", "datasets", "huggingface-hub",
                "trl", "peft", "torch", "bitsandbytes"]:
    try:
        print(package + ":", version(package))
    except PackageNotFoundError:
        print(package + ": NOT INSTALLED")

test_source = '''
import json
import torch
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer
import bitsandbytes as bnb
from datasets import load_dataset
from aligntune.core.backend_factory import create_sft_trainer
assert callable(create_sft_trainer)
assert torch.cuda.is_available(), "No CUDA GPU"
v=torch.empty((1,), device="cuda")
torch.cuda.synchronize()
print(json.dumps({"imports":"ok", "cuda":"ok", "gpu":torch.cuda.get_device_name(0),
                  "bnb_4bit":hasattr(bnb,"nn"), "trainer_factory":"ok"}))
'''
print("\nTRAINING LIBRARY IMPORT SMOKE TEST:")
smoke = subprocess.run(
    [sys.executable, "-c", test_source],
    capture_output=True, text=True, timeout=180
)
print("Exit code:", smoke.returncode)
print("stdout:\n", smoke.stdout[-4500:])
print("stderr (last 6000 chars):\n", smoke.stderr[-6000:])
TRAINING_ENV_READY = smoke.returncode == 0 and '"imports": "ok"' in smoke.stdout
print("TRAINING LIBRARIES READY:", TRAINING_ENV_READY)
if not TRAINING_ENV_READY:
    print("Do not run training. Share the smoke-test traceback for targeted repair.")
else:
    print("Core libraries import; inspect model-specific QLoRA compatibility before training.")


## 5. Dry-run the exact AlignTune training configuration (no model weights)

After validating CuratorKIT ZIP extraction and successfully importing AlignTune, run this cell to validate the 78 training records, ensure no held-out policy IDs are present, instantiate AlignTune's **TRL trainer wrapper only**, and check the LoRA + 4-bit settings. This does **not** download a model or train.

In [ ]:
import subprocess, sys, json
from pathlib import Path
train_path = CURATED / "sft_alpaca.jsonl"
assert train_path.exists(), "Run archive upload/validation first"
config_only=subprocess.run(
    [sys.executable, "-m", "scripts.check_align_config",
     "--curated-dir", str(CURATED)],
    capture_output=True, text=True, timeout=180,
)
print("AlignTune CONFIG-ONLY CHECK EXIT:", config_only.returncode)
print(config_only.stdout[-8500:])
print(config_only.stderr[-5000:])
CONFIG_VALIDATED = config_only.returncode == 0 and "config_only_valid" in config_only.stdout
print("CONFIG_VALIDATED:", CONFIG_VALIDATED)
if not CONFIG_VALIDATED:
    print("STOP. Share the error output; do not start model training.")


## 6. Initialize the quantized model and LoRA adapter — controlled load, NO training

This is the next compatibility check. It may download model weights and consume several GB of T4 memory. It calls `setup_model()` and `setup_data()` but **does not** call `train()`. We verify actual 4-bit modules, positive LoRA trainable parameters, and 78 training rows. If the T4 runs out of memory, don't start training; share the error.

In [ ]:
RUN_MODEL_LOAD = False  # Change to True after configuration-only preflight passes
MODEL_LOAD_VALIDATED = False
if RUN_MODEL_LOAD:
    assert CONFIG_VALIDATED, "Run Stage 5 and confirm CONFIG_VALIDATED=True"
    import torch, bitsandbytes as bnb
    from scripts.check_align_config import make_training_wrapper
    assert torch.cuda.is_available()
    if "trainer" in globals() and getattr(trainer, "model", None) is not None:
        raise RuntimeError("A model is already loaded. Do not load a second copy.")
    trainer = make_training_wrapper(CURATED)
    trainer.setup_model()  # Loads approved model + applies 4-bit LoRA; no training.
    model = trainer.model
    four_bit_layers = sum(isinstance(m, bnb.nn.Linear4bit) for m in model.modules())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print("4-bit linear layers:", four_bit_layers)
    print("Trainable LoRA parameters:", trainable, "of", total)
    print("GPU allocated GiB:", round(torch.cuda.memory_allocated()/1024**3, 2))
    assert four_bit_layers > 0, "4-bit quantization failed"
    assert 0 < trainable < total, "No trainable LoRA adapters detected"
    trainer.setup_data()   # Loads training data only, not 48-case holdout.
    train_rows = len(trainer.train_dataset)
    print("Prepared training records:", train_rows)
    assert train_rows > 0
    MODEL_LOAD_VALIDATED = True
    print("MODEL LOAD AND DATA SETUP OK — no training performed")
else:
    print("Model load disabled. Turn it on only after Stage 5 validates.")


## 7. Optional 10-step AlignTune training — OFF by default

Enable only after Step 6 reports `MODEL LOAD AND DATA SETUP OK`. This reuses the already initialized quantized model and dataset, avoiding a second model allocation. A successful short run would prove integration, **not** improved quality. Download the adapter and run logs before your Colab session ends.

## 7a. T4 precision guard and recovery after the FP16/BF16 gradient scaler error

On a free Colab Tesla T4, mixed precision training failed with `NotImplementedError: _amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16`. Run this diagnostic after the quantized model and 78-row training data have loaded. It upcasts **trainable LoRA parameters only**, discards any failed Trainer/optimizer state after zero completed steps, and leaves the quantized base weights unchanged. It does not train. If anything looks unexpected, stop rather than retrying blindly.

In [ ]:
from scripts.t4_precision_repair import trainable_dtype_report, repair_trainable_precision
import json, torch
assert "trainer" in globals() and trainer.model is not None
assert len(trainer.train_dataset) == 78
print("Before:", json.dumps(trainable_dtype_report(trainer.model), indent=2))
print("AlignTune model precision:", trainer.config.model.precision)
print("FP16 training flag:", trainer.config.train.fp16, "BF16 training flag:", trainer.config.train.bf16)
assert trainer.config.train.bf16 is False, "BF16 training unsupported on T4"
assert trainer.config.train.fp16 is True, "FP16 training flag expected"
repair_result = repair_trainable_precision(trainer)
print("Repair:", json.dumps(repair_result, indent=2))
PRECISION_READY = repair_result["after"]["all_trainable_are_fp32"]
print("PRECISION_READY:", PRECISION_READY, "| No model training was performed")


**Required:** Run the LoRA FP32 precision guard immediately above. Do not retry `train()` until `PRECISION_READY=True` and the previous failed Trainer has been discarded.

In [ ]:
RUN_TRAINING = False  # Explicitly change to True after model/data setup succeeded.
if RUN_TRAINING:
    import json, time, subprocess, hashlib, pathlib, traceback
    import torch, bitsandbytes as bnb
    from importlib.metadata import version
    from policyproof.core import sha256
    assert "trainer" in globals() and getattr(trainer, "model", None) is not None, "Load model first"
    assert globals().get("PRECISION_READY") is True, "Run T4 precision guard first"
    assert getattr(trainer, "train_dataset", None) is not None, "Run trainer.setup_data() first"
    assert len(trainer.train_dataset) == 78, "Unexpected training dataset"
    assert sum(isinstance(m, bnb.nn.Linear4bit) for m in trainer.model.modules()) > 0
    assert sum(p.numel() for p in trainer.model.parameters() if p.requires_grad) > 0
    assert getattr(trainer.config.train, "max_steps", None) == 10
    OUTPUT = ROOT / "artifacts" / "align_qlora_smoke"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    info_path = OUTPUT / "run_info.json"
    started = time.time()
    try:
        training_result = trainer.train()
        steps_completed = int(trainer.trainer.state.global_step)
        assert steps_completed >= 10, f"Only {steps_completed}/10 optimizer steps completed"
        adapter_dir = OUTPUT / "adapter"
        saved_dir = trainer.save_model(output_dir=str(adapter_dir))
        assert (adapter_dir / "adapter_config.json").exists(), "No adapter_config.json saved"
        weights = sorted(adapter_dir.glob("adapter_model.*"))
        assert weights, "No adapter weights saved"
        code_revision = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        run_info = {
            "status": "completed", "scope": "10-step integration smoke test only",
            "model": "CohereLabs/tiny-aya-global",
            "base_model_revision": MODEL_SHA if "MODEL_SHA" in globals() else None,
            "code_revision": code_revision,
            "curated_training_sha256": sha256(CURATED / "sft_alpaca.jsonl"),
            "train_examples": len(trainer.train_dataset),
            "steps_completed": steps_completed,
            "dataset_is_synthetic": True,
            "human_reviewed_training_data": False,
            "runtime_seconds": round(time.time() - started, 1),
            "cuda_device": torch.cuda.get_device_name(0),
            "peak_gpu_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
            "packages": {p: version(p) for p in ["aligntune", "transformers", "trl", "peft", "bitsandbytes", "torch"]},
            "adapter_files_sha256": {p.name: sha256(p) for p in weights},
            "adapter_saved_at": str(saved_dir),
            "quality_or_safety_improvement_measured": False
        }
        info_path.write_text(json.dumps(run_info, indent=2) + "\n", encoding="utf-8")
        print("TRAINING COMPLETED; steps:", steps_completed, "adapter:", saved_dir)
        print("This is not a quality or safety improvement claim.")
    except Exception as exc:
        info_path.write_text(json.dumps({
            "status": "failed", "training_executed_or_attempted": True,
            "error_type": type(exc).__name__,
            "message": str(exc)[:1200],
            "steps_completed": int(getattr(getattr(trainer, "trainer", None), "state", None).global_step)
                if getattr(getattr(trainer, "trainer", None), "state", None) is not None else None
        }, indent=2), encoding="utf-8")
        print("Training failed; original in-memory model may still exist. Do not rerun blindly.")
        raise
else:
    print("Training OFF. No optimizer steps executed.")


## 7. Export only after a successful training run. Do not label a failed or unexecuted run as successful.

In [ ]:
if "run_info" in globals() and run_info.get("status") == "completed":
    import shutil
    archive = shutil.make_archive("/content/policyproof_align_smoke", "zip", root_dir=OUTPUT)
    print("Archive:", archive)
    from google.colab import files
    files.download(archive)
else:
    print("No completed training run to export.")


## Next stage

If the install fails, share the traceback without credentials. If it works, download the adapter plus run_info evidence. Then build a fresh uninspected multilingual benchmark and verify the tuned model independently. Official docs: https://aligntune.lexsi.ai/user-guide/sft/ .